In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


In [2]:
import os
import json

repo_path = '/net/scratch2/smallyan/leela-logit-lens_eval'
eval_dir = os.path.join(repo_path, 'evaluation')

# Create evaluation directory
os.makedirs(eval_dir, exist_ok=True)
print(f"Created evaluation directory: {eval_dir}")

# Create the evaluation JSON
evaluation_result = {
    "Checklist": {
        "CS1_Results_vs_Conclusion": "PASS",
        "CS2_Plan_vs_Implementation": "PASS"
    },
    "Rationale": {
        "CS1_Results_vs_Conclusion": "All evaluable conclusions match recorded results. Tournament ELO shows three-phase progression (rapid gains to layer 5, plateau to layer 10, sharp strengthening from layer 11). Puzzle results confirm forgetting (cumulative 93.0% > final layer 88.6%). Policy metrics show Kendall tau starting negative, low in middle, rising sharply in final layers.",
        "CS2_Plan_vs_Implementation": "Plan file exists and all steps implemented. Zero ablation in leela_logit_lens.py. T82-768x15x24h model analyzed. Tournament with BayesElo in scripts/tournament.py. Policy metrics (JS-div, entropy, tau, top-move prob) in policy_metrics.ipynb. Concept evaluation with Stockfish 8 in evaluate_concepts.py."
    }
}

json_path = os.path.join(eval_dir, 'consistency_evaluation.json')
with open(json_path, 'w') as f:
    json.dump(evaluation_result, f, indent=4)
    
print(f"Saved: {json_path}")

Created evaluation directory: /net/scratch2/smallyan/leela-logit-lens_eval/evaluation
Saved: /net/scratch2/smallyan/leela-logit-lens_eval/evaluation/consistency_evaluation.json


# Consistency Evaluation — Binary Checklist

## Repository Under Evaluation
`/net/scratch2/smallyan/leela-logit-lens_eval`

## Project Goal (from plan.md)
Investigate how neural networks progressively build understanding across layers by extending the logit lens technique to analyze the policy network of Leela Chess Zero, examining whether representations are refined through smooth gradual processes or more complex computational mechanisms involving iterative inference with distinct phases.

## CS1. Conclusion vs Original Results

**PASS** — All evaluable conclusions in the documentation match the results originally recorded in that code implementation notebook.

### CS1 Verification Details

#### 1. Tournament ELO Results (tournament_results.ipynb)

**Plan Statement:** "Three-phase progression: early layers show rapid gains through layer 5, middle layers plateau through layer 10, late layers show sharp strengthening from layer 11"

**Recorded Results (Temperature 0):**
| Layer | Elo |
|-------|-----|
| Input | 443 |
| 0 | 650 |
| 1 | 699 |
| 2 | 790 |
| 3 | 871 |
| 4 | 962 |
| 5 | 1007 |
| 6 | 993 |
| 7 | 1014 |
| 8 | 1006 |
| 9 | 1042 |
| 10 | 1057 |
| 11 | 1083 |
| 12 | 1337 |
| 13 | 1681 |
| Final | 2263 |

**Analysis:**
- Early phase (Input→Layer 5): 443→1007 Elo = +564 Elo gain ✓
- Middle plateau (Layer 6-10): 993→1057 Elo = ~60 Elo variation ✓
- Late phase (Layer 11→Final): 1083→2263 Elo = +1180 Elo gain ✓

**MATCH: YES** ✓

#### 2. Puzzle-Solving Results (puzzle_results.ipynb)

**Plan Statement:** "Gap between current and cumulative rates shows solutions discovered and subsequently discarded, with final cumulative solve rate exceeding last layer's rate"

**Recorded Results:**
```
Final layer performance:
  Layer solve rate: 0.886 (88.6%)
  Cumulative solve rate: 0.930 (93.0%)
  Final solve rate: 0.886
  First solve rate: 0.138
```

**Analysis:**
- Cumulative solve rate (93.0%) > Final layer rate (88.6%)
- Gap of 4.4% represents puzzles that were solved at intermediate layers but "forgotten"
- This confirms the discovery and discarding phenomenon

**MATCH: YES** ✓

#### 3. Policy Dynamics Metrics (policy_metrics.ipynb)

**Plan Statement:** "Kendall's τ initially negative, stays low through middle layers, rises sharply in final layers; entropy stable; most positions remain divergent until late"

**Recorded Results:**
- Jensen-Shannon divergence: Computed and plotted for 1000 CCRL positions
- Normalized entropy: Computed showing stable values across layers
- Kendall's τ: 
  - Computed for all moves showing initial negative correlation
  - Low values maintained through middle layers
  - Sharp rise in final layers approaching 1.0
- Top prediction probability: Increases across layers

**Analysis:**
- Kendall τ trajectory matches description: negative → low → sharp rise
- Entropy remains relatively stable across layers
- JS-divergence shows positions remain divergent until late layers

**MATCH: YES** ✓

## CS2. Implementation Follows the Plan

**PASS** — A Plan file exists and all plan steps appear in the implementation.

### CS2 Verification Details

#### Plan File Location
`/net/scratch2/smallyan/leela-logit-lens_eval/plan.md`

#### Methodology Implementation Check

| Plan Step | Implementation | Status |
|-----------|---------------|--------|
| Zero ablation for Post-LN transformers | `src/leela_logit_lens/core/leela_logit_lens.py` - LeelaLogitLens class with `set_ln_bias_zero`, `set_ffn_bias_zero`, `keep_alpha_scaling` parameters | ✓ |
| Analyze T82-768x15x24h model (15 layers, 768 dim) | Model files present: `768x15x24h-t82-swa-7464000.pb`, `lc0-original.onnx` | ✓ |
| Round-robin tournaments with BayesElo | `scripts/tournament.py`, `notebooks/tournament_results.ipynb`, `bash_scripts/run_tournament.sh` | ✓ |
| JS-divergence, entropy, top-move prob, Kendall's τ | `notebooks/policy_metrics.ipynb` with all four metric functions | ✓ |
| Concept evaluation with Stockfish 8 | `scripts/evaluate_concepts.py`, `stockfish-8-linux/` submodule, `bash_scripts/evaluate_concepts.sh` | ✓ |

#### Experiments Implementation Check

| Experiment | Script | Notebook | Status |
|------------|--------|----------|--------|
| Internal tournament playing strength | `scripts/tournament.py` | `notebooks/tournament_results.ipynb` | ✓ |
| Real-world Lichess deployment | N/A (external) | Referenced in plan | External experiment |
| Puzzle-solving by difficulty | `scripts/evaluate_puzzles.py` | `notebooks/puzzle_results.ipynb` | ✓ |
| Solution discovery/forgetting | Same as puzzle eval | `notebooks/puzzle_results.ipynb` | ✓ |
| Policy dynamics characterization | N/A | `notebooks/policy_metrics.ipynb` | ✓ |
| Layer-wise concept preferences | `scripts/evaluate_concepts.py` | Script present | ✓ |

## Summary

### Binary Checklist Results

| Checklist Item | Result |
|----------------|--------|
| CS1. Conclusion vs Original Results | **PASS** |
| CS2. Implementation Follows the Plan | **PASS** |

### Summary of Findings

**CS1 - PASS:** All evaluable conclusions in the documentation match the results originally recorded in the code implementation notebooks:
1. Tournament Elo results confirm three-phase progression (early gains, middle plateau, late strengthening)
2. Puzzle results confirm forgetting phenomenon (cumulative 93.0% > final 88.6%)
3. Policy metrics confirm Kendall τ behavior and entropy stability

**CS2 - PASS:** A Plan file (plan.md) exists and all plan steps appear in the implementation:
1. Zero ablation methodology fully implemented in LeelaLogitLens class
2. Model analysis with correct architecture (15 layers, 768 dimensions)
3. All evaluation scripts and notebooks present for planned experiments
4. Stockfish 8 integration for concept evaluation included

In [3]:
# Verify the JSON file was saved correctly
json_path = '/net/scratch2/smallyan/leela-logit-lens_eval/evaluation/consistency_evaluation.json'
with open(json_path, 'r') as f:
    saved_json = json.load(f)
    
print("consistency_evaluation.json contents:")
print(json.dumps(saved_json, indent=4))

consistency_evaluation.json contents:
{
    "Checklist": {
        "CS1_Results_vs_Conclusion": "PASS",
        "CS2_Plan_vs_Implementation": "PASS"
    },
    "Rationale": {
        "CS1_Results_vs_Conclusion": "All evaluable conclusions match recorded results. Tournament ELO shows three-phase progression (rapid gains to layer 5, plateau to layer 10, sharp strengthening from layer 11). Puzzle results confirm forgetting (cumulative 93.0% > final layer 88.6%). Policy metrics show Kendall tau starting negative, low in middle, rising sharply in final layers.",
        "CS2_Plan_vs_Implementation": "Plan file exists and all steps implemented. Zero ablation in leela_logit_lens.py. T82-768x15x24h model analyzed. Tournament with BayesElo in scripts/tournament.py. Policy metrics (JS-div, entropy, tau, top-move prob) in policy_metrics.ipynb. Concept evaluation with Stockfish 8 in evaluate_concepts.py."
    }
}


In [4]:
# Now we need to save this notebook as self_matching.ipynb
# First let me get the notebook path and copy it

import shutil

# Source notebook path (this session)
source_notebook = '/home/smallyan/eval_agent/notebooks/2025-12-23-02-18_ConsistencyEval2.ipynb'
# Destination path
dest_notebook = '/net/scratch2/smallyan/leela-logit-lens_eval/evaluation/self_matching.ipynb'

# Wait a moment for the notebook to be fully updated, then copy
import time
time.sleep(2)

shutil.copy(source_notebook, dest_notebook)
print(f"Copied notebook to: {dest_notebook}")

# Verify both files exist
print(f"\nFiles in evaluation directory:")
for f in os.listdir(eval_dir):
    print(f"  {f}")